### Imports

In [ ]:
import os
from datetime import datetime

import pandas
import pycountry_convert
from sklearn.metrics import mean_absolute_percentage_error
from xgboost import XGBRegressor

In [ ]:
os.makedirs("./data/income/", exist_ok=True)

### Get worldbank classification and add it to the dataset
See https://datahelpdesk.worldbank.org/knowledgebase/articles/906519-world-bank-country-and-lending-groups

In [ ]:
worldbank_data = pandas.read_csv("./data/income/worldbank_classification.csv")

In [ ]:
worldbank_data["country_code"] = [
    pycountry_convert.country_alpha3_to_country_alpha2(code)
    for code in worldbank_data["Code"]
]

In [ ]:
# Adjust XKX due to error
worldbank_data.loc[len(worldbank_data)] = ["XKX", "Upper middle income", "XK"]

In [ ]:
worldbank_data = worldbank_data.dropna()
worldbank_data.head()

In [ ]:
dataset = pandas.read_parquet(
    "./data/2025-09-26-0752_total_dataset.parquet", engine="pyarrow"
)

In [ ]:
dataset["country_code"] = [
    code.split("_")[0] for code in dataset["region_code"]
]

In [ ]:
dataset["continent_code"] = [
    pycountry_convert.country_alpha2_to_continent_code(code)
    for code in dataset["country_code"]
]

In [ ]:
worldbank_data["Income group"].unique()

In [ ]:
dataset = dataset.merge(worldbank_data, on="country_code")

### Split dataset into continent

In [ ]:
list_income_group_datasets = []

for income_group, group in dataset.groupby("Income group"):
    list_income_group_datasets.append([income_group, group.copy(deep=True)])

In [ ]:
for entry in list_income_group_datasets:
    print(
        entry[0],
        "has",
        len(entry[1]),
        "observations, for",
        len(entry[1]["country_code"].unique()),
        "countries in the continent.",
    )

### Functions

In [ ]:
def compute_train_val_test(cur_dataset):
    """
    Split the dataset into test, train, and validation sets.

    Parameters
    ----------
    cur_dataset (pandas.DataFrame): The dataset to split.

    Returns
    -------
    pandas.DataFrame: The train set.
    pandas.DataFrame: The validation set.
    pandas.DataFrame: The test set.
    """
    # Initialize empty dataframes for test and validation sets
    test_set = pandas.DataFrame()
    test_set_indices = []
    validation_set = pandas.DataFrame()
    validation_set_indices = []

    for name, group in cur_dataset.groupby("region_code"):
        # Keep track of the last available year for each region
        max_year = group["local_year"].max()

        # Select the test set by selecting the last year
        group_test_set = group[group["local_year"] == max_year].copy()
        test_set_indices.append(group_test_set.index)
        test_set = pandas.concat([test_set, group_test_set], ignore_index=True)

        # Select the validation set by selecting the second last year
        group_val_set = group[group["local_year"] == max_year - 1].copy()
        validation_set_indices.append(group_val_set.index)
        validation_set = pandas.concat(
            [validation_set, group_val_set], ignore_index=True
        )

    print(
        "Test set size:",
        round((len(test_set) / len(cur_dataset)) * 100, 2),
        "% of total dataset",
    )
    print(
        "Validation set size:",
        round((len(validation_set) / len(cur_dataset)) * 100, 2),
        "% of total dataset",
    )
    # Obtain the indicies of the test and validation sets
    all_test_set_indices = [
        index for list_indicies in test_set_indices for index in list_indicies
    ]
    all_val_set_indices = [
        index
        for list_indicies in validation_set_indices
        for index in list_indicies
    ]
    # Drop test and validation sets from the current dataset
    train_set = cur_dataset.drop(index=all_test_set_indices).copy()
    train_set = train_set.drop(index=all_val_set_indices)

    val_set = cur_dataset.loc[all_val_set_indices]
    test_set = cur_dataset.loc[all_test_set_indices]

    return train_set, val_set, test_set

In [ ]:
def prepare_data(dataset: pandas.DataFrame):
    """
    Process the dataset into splits to be used in training the model.

    Returns
    -------
    features : pandas.DataFrame
        Features for the model.
    target : pandas.Series
        Column with the target variable.
    groups : pandas.Series
        Column containing the region codes
    """
    features = (
        dataset[
            [
                "local_hour",
                "is_weekend",
                "local_month",
                "year_temp_top1",
                "year_temp_top3",
                "monthly_temp_avg_top1",
                "monthly_temp_avg_rank_top1",
                "year_temp_avg_top1",
                "year_temp_percentile_5",
                "year_temp_percentile_95",
                # "year_electricity_demand_per_capita_mwh",
                # "year_gdp",
            ]
        ]
        .copy(deep=True)
        .reset_index(drop=True)
    )

    categorical_features = [
        "local_hour",
        "is_weekend",
        "local_month",
        "monthly_temp_avg_rank_top1",
    ]

    for cat_feature in categorical_features:
        features[cat_feature] = features[cat_feature].astype("category")

    target = (
        dataset["load_mw_percentage"].copy(deep=True).reset_index(drop=True)
    )
    groups = dataset["region_code"].copy(deep=True).reset_index(drop=True)

    return features, target, groups

In [ ]:
def calculate_test_error_metric(
    error_metric,
    error_metric_name: str,
    current_predictions,
    current_target,
    current_groups,
    output_folder,
    message: str = "",
) -> pandas.DataFrame:
    """
    Calcuate the mean absolute percentage error for the test set.

    Saves the results to a parquet and CSV file.

    Returns
    -------
    pandas.DataFrame
        A DataFrame with the region codes, years,
        and mean absolute percentage errors for the test set.
    """
    list_test_metric_values = []
    for name, group in pandas.DataFrame(current_groups).groupby("region_code"):
        current_metric = error_metric(
            current_predictions[group.index], current_target.iloc[group.index]
        )

        list_test_metric_values.append([name, current_metric])

    df_validation_metric_values = pandas.DataFrame(
        list_test_metric_values, columns=["region_code", error_metric_name]
    )

    df_validation_metric_values.to_parquet(
        output_folder
        + datetime.now().strftime("%Y-%m-%d-%H%M")
        + "_"
        + error_metric_name
        + "_values"
        + message
        + ".parquet",
        engine="pyarrow",
    )
    df_validation_metric_values.to_csv(
        output_folder
        + datetime.now().strftime("%Y-%m-%d-%H%M")
        + "_"
        + error_metric_name
        + "_values"
        + message
        + ".csv"
    )

    return df_validation_metric_values

In [ ]:
"_".join(income_group.lower().split())

### Train a model per continent

In [ ]:
for income_group, group in list_income_group_datasets:
    income_group_name = "_".join(income_group.lower().split())
    print(income_group_name)

    train_set, validation_set, test_set = compute_train_val_test(group)

    # Generate the train, validation, and test datasets
    train_features, train_target, train_groups = prepare_data(train_set)
    val_features, val_target, val_groups = prepare_data(validation_set)
    test_features, test_target, test_groups = prepare_data(test_set)

    # Initialize the XGBoost regressor
    xgb_model = XGBRegressor(
        random_state=42,
        enable_categorical=True,
        eval_metric=mean_absolute_percentage_error,
    )

    # Train the model
    xgb_model.fit(
        train_features, train_target, eval_set=[(val_features, val_target)]
    )

    # Store the trained model
    xgb_model.save_model(
        "./data/income/"
        + datetime.now().strftime("%Y-%m-%d-%H%M")
        + "_xgboost_model_"
        + income_group_name
        + ".bin"
    )

    # Predict on test set and calculate MAPE
    test_predictions = xgb_model.predict(test_features)
    calculate_test_error_metric(
        mean_absolute_percentage_error,
        "MAPE",
        test_predictions,
        test_target,
        test_groups,
        "data/income/",
        "_".join(("", income_group_name, "test")),
    )
    # Predict on validation set and calculate MAPE
    val_predictions = xgb_model.predict(val_features)
    calculate_test_error_metric(
        mean_absolute_percentage_error,
        "MAPE",
        val_predictions,
        val_target,
        val_groups,
        "data/income/",
        "_".join(("", income_group_name, "val")),
    )
    # Predict on training set and calculate MAPE
    train_predictions = xgb_model.predict(train_features)
    calculate_test_error_metric(
        mean_absolute_percentage_error,
        "MAPE",
        train_predictions,
        train_target,
        train_groups,
        "data/income/",
        "_".join(("", income_group_name, "train")),
    )